In [ ]:
!pip install datasets
!pip install tqdm

In [ ]:
import datasets
import pandas as pd
import tqdm.notebook as tqdm

In [ ]:
# derived from documentation for google/smol dataset
languages= {
    # 'af': 'afr',
    # 'lg': 'lug',
    # 'ach': 'ach',
    # 'alz': 'alz',
    # 'cgg': 'cgg',

    # New Languages
    'af': 'afr',
    'ak': 'aka',
    'am': 'amh',
    'bem': 'bem',
    'ber': 'ber',
    'bm': 'bar',
    'din': 'din',
    'ee': 'ewe',
    'ff': 'ful',
    'fr': 'fra',
    'ha': 'hau',
    'ig': 'ibo',
    'ki': 'kik',
    'kr': 'kau',
    'ln': 'lin',
    'luo': 'luo',
    'mg': 'mlg',
    'nr': 'nbl',
    'ny': 'nya',
    'om': 'orm',
    'pcm': 'pcm',
    'rn': 'run',
    'rw': 'kin',
    'sn': 'sna',
    'so': 'som',
    'st': 'sot',
    'sw': 'swh',
    'tn': 'tsn',
    'wo': 'wol',
    'xh': 'xho',
    'yo': 'yor',
    'zu': 'zul',

}
repo_id = 'Sunbird/external-translation-data'

In [ ]:
failed_languages = []

In [ ]:

for lang, iso_lang in tqdm.tqdm(languages.items(), desc="Processing languages"):
    sources = []
    targets = []

    # gatitos
    try:
        gatitos = datasets.load_dataset('google/smol', f'gatitos__en_{lang}', split='train')
        for example in gatitos:
            if example['sl'] != 'en':
                raise ValueError(f'Unexpected language direction: {example}')
            for tgt in example['trgs']:
                sources.append(example['src'])
                targets.append(tgt)
        print(f"  gatitos: {len(sources)} examples so far")
    except Exception as e:
        print(f"  Skipping gatitos__{lang}: {e}")

    #smolsent
    try:
        smolsent = datasets.load_dataset('google/smol', f'smolsent__en_{lang}', split='train')
        for example in smolsent:
            if example['sl'] != 'en':
                raise ValueError(f'Unexpected language direction: {example}')
            sources.append(example['src'])
            targets.append(example['trg'])
        print(f"  smolsent: {len(sources)} examples so far")
    except Exception as e:
        print(f"  Skipping smolsent__{lang}: {e}")

    # smoldoc
    try:
        smoldoc = datasets.load_dataset('google/smol', f'smoldoc__en_{lang}', split='train')
        for example in smoldoc:
            if example['sl'] != 'en':
                raise ValueError(f'Unexpected language direction: {example}')
            sources.append(' '.join(example['srcs']))
            targets.append(' '.join(example['trgs']))
        print(f"  smoldoc: {len(sources)} examples total")
    except Exception as e:
        print(f"  Skipping smoldoc__{lang}: {e}")


    if not sources:
        print(f"  No data found for {lang}, skipping push.")
        failed_languages.append((lang, iso_lang, "No data found across all configs"))
        continue

    df = pd.DataFrame({
        'eng_text': sources,
        f'{iso_lang}_text': targets,
    })

    try:
        ds = datasets.Dataset.from_pandas(df)
        ds.push_to_hub(
            repo_id=repo_id,
            config_name=f'google_smol_{iso_lang}',
            private=True,
            split='train',
        )
        print(f"  Pushed {len(df)} rows as google_smol_{iso_lang}")
    except Exception as e:
        print(f"  ERROR pushing {lang} ({iso_lang}): {e}")
        failed_languages.append((lang, iso_lang, str(e)))

# Summary
print(f"Processed {len(languages) - len(failed_languages)}/{len(languages)} languages successfully.")
if failed_languages:
    print("Failed languages:")
    for lang, iso_lang, err in failed_languages:
        print(f"  {lang} ({iso_lang}): {err}")